# Telco Customer Churn Assignment

## Environment Setup and Project Specs

- Python: 3.11+
- Core packages: pandas, numpy, matplotlib, seaborn, scikit-learn, joblib, fastapi, uvicorn, pytest
- Input files:
  - `../Data Science Assignment.pdf`
  - `../TelcoCustomerChurn.csv`
  - `../TelcoCustomerChurn - Data Dictionary.csv`
- Output locations:
  - `../model/churn_model.joblib`
  - `../model/training_summary.json`
  - `../results/` for exported figures and tables
- Shared implementation: `../churn_pipeline.py`

This notebook mirrors the assignment requirements and reuses the shared pipeline so preprocessing, training, and API prediction stay consistent.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
NOTEBOOK_DIR = PROJECT_ROOT / "customer_churn_project" / "notebook" if (PROJECT_ROOT / "customer_churn_project").exists() else PROJECT_ROOT / "notebook"
PROJECT_DIR = NOTEBOOK_DIR.parent
RESULTS_DIR = PROJECT_DIR / "results"
MODEL_DIR = PROJECT_DIR / "model"
DATA_PATH = PROJECT_ROOT / "TelcoCustomerChurn.csv"
DICTIONARY_PATH = PROJECT_ROOT / "TelcoCustomerChurn - Data Dictionary.csv"
ASSIGNMENT_PATH = PROJECT_ROOT / "Data Science Assignment.pdf"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_packages = ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "joblib", "fastapi"]
package_status = {}
for package in required_packages:
    try:
        __import__(package)
        package_status[package] = "installed"
    except Exception:
        package_status[package] = "missing"

print("Package status:")
display(pd.Series(package_status))

# The notebook imports the shared implementation used by training and the API.
from churn_pipeline import (
    DATA_PATH as DEFAULT_DATA_PATH,
    DATA_DICTIONARY_PATH as DEFAULT_DICTIONARY_PATH,
    EXPECTED_INPUT_COLUMNS,
    MODEL_PATH,
    SUMMARY_PATH,
    FeatureEngineer,
    build_pipeline,
    clean_dataframe,
    compare_configs,
    evaluate_predictions,
    fit_final_model,
    load_data,
    save_artifact,
    split_data,
)

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data file: {DATA_PATH}")
print(f"Dictionary file: {DICTIONARY_PATH}")
print(f"Assignment PDF: {ASSIGNMENT_PATH}")

## Load the Assignment PDF and CSV Files

This section reads the assignment PDF and the provided CSVs, then captures the requirements in a traceable format so each notebook section can be linked back to the brief.

In [ ]:
def read_assignment_text(pdf_path: Path) -> str:
    if shutil.which("pdftotext"):
        txt_path = pdf_path.with_suffix(".txt")
        subprocess.run(["pdftotext", str(pdf_path), str(txt_path)], check=True)
        return txt_path.read_text(encoding="utf-8", errors="ignore")
    try:
        from pypdf import PdfReader
    except Exception as exc:
        raise RuntimeError("Install pypdf or ensure pdftotext is available") from exc
    reader = PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

for path in [ASSIGNMENT_PATH, DATA_PATH, DICTIONARY_PATH]:
    print(f"{path.name}: {'found' if path.exists() else 'missing'}")

assignment_text = read_assignment_text(ASSIGNMENT_PATH)
print(assignment_text[:3000])

churn_df = pd.read_csv(DATA_PATH)
dictionary_df = pd.read_csv(DICTIONARY_PATH)
print(churn_df.shape)
print(dictionary_df.shape)

display(churn_df.head(3))
display(dictionary_df.head(10))

requirements_map = pd.DataFrame(
    [
        {"assignment_requirement": "Data understanding and preparation", "notebook_section": "Inspect CSV Structure and Validate Schemas / Clean and Prepare the Data"},
        {"assignment_requirement": "Exploratory data analysis", "notebook_section": "Perform Required Exploratory Analysis"},
        {"assignment_requirement": "Feature engineering", "notebook_section": "Engineer Features Required by the Assignment"},
        {"assignment_requirement": "Decision tree modeling", "notebook_section": "Build the Required Analysis or Model Pipeline"},
        {"assignment_requirement": "Evaluation and interpretation", "notebook_section": "Evaluate Results Against the Requirements"},
        {"assignment_requirement": "Saved model and REST API", "notebook_section": "Export Outputs, Figures, and Submission Files"},
    ]
)
display(requirements_map)

## Inspect CSV Structure and Validate Schemas

This section checks row counts, column names, missing values, duplicate records, and column types against the data dictionary before any modeling or charting work.

In [ ]:
expected_columns = list(dictionary_df["Column"].astype(str))
observed_columns = list(churn_df.columns)
missing_columns = [column for column in expected_columns if column not in observed_columns]
extra_columns = [column for column in observed_columns if column not in expected_columns]

schema_report = pd.DataFrame(
    {
        "column": observed_columns,
        "dtype": churn_df.dtypes.astype(str).values,
        "missing_values": churn_df.isna().sum().values,
    }
)

print(f"Rows: {len(churn_df):,}")
print(f"Columns: {len(churn_df.columns)}")
print(f"Missing schema columns: {missing_columns}")
print(f"Extra schema columns: {extra_columns}")
print(f"Duplicate rows: {churn_df.duplicated().sum()}")
display(schema_report)
display(churn_df[TARGET_COLUMN].value_counts(dropna=False).rename_axis("Churn").reset_index(name="count"))

clean_df = clean_dataframe(churn_df)
print(clean_df.isna().sum().sort_values(ascending=False).head(10))
display(clean_df.head(3))

## Perform Required Exploratory Analysis

This section creates the required visual checks: churn distribution, service and contract comparisons, numerical distributions, and a correlation view.

In [ ]:
eda_df = clean_df.copy()
eda_df["service_count"] = (
    eda_df[["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]]
    .astype(str)
    .apply(lambda col: col.eq("Yes").astype(int))
    .sum(axis=1)
)
eda_df["avg_monthly_charge_per_tenure"] = eda_df["MonthlyCharges"] / eda_df["tenure"].replace(0, np.nan)
eda_df["is_month_to_month"] = (eda_df["Contract"] == "Month-to-month").astype(int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.countplot(data=eda_df, x="Churn", ax=axes[0, 0])
axes[0, 0].set_title("Churn Distribution")

sns.boxplot(data=eda_df, x="Churn", y="tenure", ax=axes[0, 1])
axes[0, 1].set_title("Tenure by Churn")

sns.boxplot(data=eda_df, x="Churn", y="MonthlyCharges", ax=axes[0, 2])
axes[0, 2].set_title("Monthly Charges by Churn")

sns.countplot(data=eda_df, x="Contract", hue="Churn", ax=axes[1, 0])
axes[1, 0].set_title("Contract Type vs Churn")
axes[1, 0].tick_params(axis="x", rotation=20)

sns.histplot(data=eda_df, x="service_count", hue="Churn", multiple="stack", bins=8, ax=axes[1, 1])
axes[1, 1].set_title("Service Count vs Churn")

corr_source = eda_df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "service_count", "avg_monthly_charge_per_tenure", "is_month_to_month"]].copy()
sns.heatmap(corr_source.corr(numeric_only=True), annot=True, cmap="Blues", ax=axes[1, 2])
axes[1, 2].set_title("Numeric Feature Correlation")

plt.tight_layout()
plt.show()

print("Business insight notes:")
print("- Churn is typically concentrated in month-to-month customers.")
print("- Short-tenure customers and higher monthly charges are usually linked to churn risk.")
print("- Customers with more services often show lower churn, suggesting bundle stickiness.")

## Engineer Features Required by the Assignment

The shared pipeline already creates two reusable features: `service_count` and `avg_monthly_charge_per_tenure`, plus a month-to-month contract flag used during modeling.

In [ ]:
feature_df = clean_df.copy()
feature_df["service_count"] = (
    feature_df[["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]]
    .astype(str)
    .apply(lambda col: col.eq("Yes").astype(int))
    .sum(axis=1)
)
feature_df["avg_monthly_charge_per_tenure"] = feature_df["MonthlyCharges"] / feature_df["tenure"].replace(0, np.nan)
feature_df["is_month_to_month"] = (feature_df["Contract"] == "Month-to-month").astype(int)

feature_engineered_preview = feature_df[["service_count", "avg_monthly_charge_per_tenure", "is_month_to_month"]].head(10)
display(feature_engineered_preview)

print("Feature notes:")
print("- service_count counts active value-added services to capture bundle depth.")
print("- avg_monthly_charge_per_tenure approximates intensity of spend relative to customer age.")
print("- is_month_to_month flags the contract pattern most often associated with churn.")

model_train = feature_df.drop(columns=["Churn"])
model_target = feature_df["Churn"]
model_split = split_data(feature_df)

configs = {
    "baseline": {"max_depth": None, "min_samples_leaf": 1, "class_weight": None},
    "constrained": {"max_depth": 6, "min_samples_leaf": 20, "class_weight": "balanced"},
}
comparisons = compare_configs(model_split["X_train"], model_split["y_train"], model_split["X_validation"], model_split["y_validation"], configs)
comparison_df = pd.DataFrame([
    {"config_name": item["config_name"], **item["metrics"]} for item in comparisons
])
display(comparison_df)

best = comparisons[0]
final_model = fit_final_model(model_split["X_train_full"], model_split["y_train_full"], best["params"])
final_predictions = final_model.predict(model_split["X_test"])
final_metrics = evaluate_predictions(model_split["y_test"], final_predictions)

display(pd.Series(final_metrics, name="final_metrics"))
print("Final selection:", best["config_name"], best["params"])

artifact_payload = {
    "pipeline": final_model,
    "selected_config_name": best["config_name"],
    "selected_config": best["params"],
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
save_artifact(MODEL_PATH, final_model, best["config_name"], best["params"])

artifact_preview = {"model_path": str(MODEL_PATH), "summary_path": str(SUMMARY_PATH)}
display(pd.Series(artifact_preview, name="exports"))

sample_payload = {column: clean_df.iloc[0][column] for column in EXPECTED_INPUT_COLUMNS if column in clean_df.columns}
display(pd.Series(sample_payload, name="sample_request"))
print("POST /predict will return a churn label and churn probability when supplied with the sample_request fields.")

## Evaluate Results and Clear Run Steps

Run the notebook top to bottom after placing the three source files in the workspace root. The final section below summarizes the execution order and submission artifacts.

In [ ]:
evaluation_summary = pd.DataFrame(
    [
        {"requirement": "Train/test split", "status": "met", "evidence": "70/30 split with random_state=42 via split_data()"},
        {"requirement": "Decision tree comparison", "status": "met", "evidence": "baseline vs constrained configs compared"},
        {"requirement": "Model evaluation", "status": "met", "evidence": "accuracy, precision, recall, F1, confusion matrix computed"},
        {"requirement": "Model saving", "status": "met", "evidence": f"artifact saved to {MODEL_PATH}"},
        {"requirement": "REST API", "status": "met", "evidence": "app.py exposes POST /predict"},
    ]
)
display(evaluation_summary)

run_steps = [
    "1. Open this notebook from customer_churn_project/notebook.",
    "2. Make sure Data Science Assignment.pdf, TelcoCustomerChurn.csv, and TelcoCustomerChurn - Data Dictionary.csv stay one directory above the notebook.",
    "3. Install requirements: pip install -r customer_churn_project/requirements.txt.",
    "4. Run the notebook top to bottom.",
    "5. If you want the reusable model artifact, run python customer_churn_project/train.py from the project root.",
    "6. Start the API with uvicorn app:app --reload from customer_churn_project/.",
    "7. Test the endpoint with customer_churn_project/sample_request.json.",
    "8. Check customer_churn_project/model/ for churn_model.joblib and training_summary.json.",
]
print("\n".join(run_steps))

submission_files = pd.DataFrame(
    [
        {"file": "customer_churn_project/churn_pipeline.py", "purpose": "shared preprocessing, modeling, and persistence"},
        {"file": "customer_churn_project/train.py", "purpose": "train and save the final decision tree pipeline"},
        {"file": "customer_churn_project/app.py", "purpose": "FastAPI /predict endpoint"},
        {"file": "customer_churn_project/tests/", "purpose": "pytest specs for the pipeline and API"},
        {"file": "customer_churn_project/requirements.txt", "purpose": "project dependencies"},
        {"file": "customer_churn_project/sample_request.json", "purpose": "example API payload"},
    ]
)
display(submission_files)